In [34]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

In [2]:
# load PDF
PDF_PATH = "../data/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading PDF for file: ../data/crag_paper.pdf ...


Loaded 254 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 6
  ListItem: 41
  NarrativeText: 107
  Table: 7
  Title: 32
  UncategorizedText: 52


In [17]:
elements[0].metadata.keys()

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'file_directory', 'filename', 'category', 'element_id'])

In [18]:
for element in elements:
    if element.metadata.get("category") == "Image":
        print(element.metadata.keys())
        print(element.metadata)
        break

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'image_path', 'file_directory', 'filename', 'category', 'element_id'])
{'source': '../data/crag_paper.pdf', 'coordinates': {'points': ((np.float64(372.93637515555537), np.float64(2949.874959546666)), (np.float64(372.93637515555537), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(2949.874959546666))), 'system': 'PixelSpace', 'layout_width': 2894, 'layout_height': 4093}, 'last_modified': '2026-02-17T00:01:25', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'image_path': 'd:\\rag-multimodal\\notebook\\figures\\figure-1-1.jpg', 'file_directory': '../data', 'filename': 'crag_paper.pdf', 'category': 'Image', 'element_id': '0eda6d21a6f5ee087ef5f507aaa8896c'}


In [20]:
# caption images with VLM
vlm = ChatOpenAI(model="gpt-5-mini")

IMAGE_CAPTION_SYSTEM_PROMPT = """You are a document analysis assistant. Your task is to generate \
detailed, accurate descriptions of images extracted from a document. These descriptions will be \
embedded into a vector store and used for semantic retrieval, so they must capture all information \
a user might search for.

For each image, describe:
- The image type (chart, diagram, photograph, table, illustration, screenshot, etc.)
- All visible text, labels, titles, captions, and annotations
- Key data, values, trends, or patterns (especially for charts and graphs)
- The main subject and all important visual elements
- Spatial relationships and structure where relevant

Be specific and thorough. Avoid vague language."""

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def caption_image(image_path: str) -> str:
    b64 = encode_image(image_path)
    messages = [
        SystemMessage(content=IMAGE_CAPTION_SYSTEM_PROMPT),
        HumanMessage(content=[
            {"type": "text", "text": "Describe this image extracted from a document."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]),
    ]
    return vlm.invoke(messages).content

In [21]:
image_docs = []

for el in elements:
    if el.metadata.get("category") == "Image":
        image_path = el.metadata.get("image_path", "")
        if image_path:
            caption = caption_image(image_path)
            image_docs.append(Document(page_content=caption, metadata=el.metadata))

print(f"Captioned {len(image_docs)} images")

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Captioned 6 images


In [24]:
print(image_docs[-1].page_content)

- Image type: 2-series line chart (scatter points connected by lines) with markers and gridlines.

- Overall subject / purpose: Shows how "Accuracy of generation" (y-axis) varies as a function of "Accuracy of retrieval" (x-axis) for two methods named Self-RAG and Self-CRAG.

- Axes and scale:
  - X-axis label: "Accuracy of retrieval".
  - X-axis tick labels (left to right): "69.8 (Actual)", 60, 50, 40, 30, 20, 10. Vertical dashed gridlines align with each x tick.
  - Y-axis label: "Accuracy of generation".
  - Y-axis numeric range shown from 20 up to 70 (tick marks at roughly 20, 30, 40, 50, 60, 70).

- Legend: located inside the plot area near the top-right. Contains:
  - green star marker + text "Self-RAG"
  - gray diamond marker + text "Self-CRAG"

- Series, markers and colors:
  - Self-RAG: green line with green five-pointed star markers.
  - Self-CRAG: gray line with gray diamond markers.

- All visible text and annotations:
  - Axis labels as above.
  - The legend entries "Self-R

In [79]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]

text_docs = splitter.split_documents(text_elements)
caption_docs = splitter.split_documents(image_docs)

all_docs = text_docs + caption_docs
print(f"Total chunks: {len(all_docs)} ({len(text_docs)} text + {len(caption_docs)} captions)")

Total chunks: 277 (257 text + 20 captions)


In [80]:
# embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [81]:
# vector store
vector_store = Chroma.from_documents(filter_complex_metadata(all_docs), embeddings)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [82]:
# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [83]:
# RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"),
])

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    messages = prompt.format_messages(context=text_context, question=question)

    if image_chunks:
        seen_paths = set()
        image_content = []
        for doc in image_chunks:
            image_path = doc.metadata.get("image_path")
            if image_path and image_path not in seen_paths:
                seen_paths.add(image_path)
                image_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
                })

        if image_content:
            text_content = messages[-1].content
            messages[-1] = HumanMessage(content=[
                {"type": "text", "text": text_content},
                *image_content,
            ])

    return messages

llm = ChatOpenAI(model="gpt-5-mini")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | {"response": llm | StrOutputParser(), "context": RunnablePassthrough()}
)

In [84]:
# test
question = "How does Self-CRAG compares with Self-RAG as shown in the line chart. Can you explain this in a little bit more detail?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Summary: Self-CRAG consistently outperforms Self-RAG at every retrieval-accuracy point and is noticeably more robust as retrieval quality drops.

Details from the chart:
- At the actual retrieval accuracy (~69.8%), Self-CRAG ≈ 62–63% generation accuracy vs Self-RAG ≈ 55% (gap ≈ 7–8 points).
- At mid-level retrieval (50–40%), Self-CRAG ≈ 58–60% while Self-RAG ≈ 46–42% (gap ≈ 12–16 points).
- At low retrieval (10%), Self-CRAG ≈ 52% vs Self-RAG ≈ 33% (gap ≈ 19 points).

Trend: both methods lose generation accuracy as retrieval accuracy decreases, but Self-RAG’s decline is much steeper. Self-CRAG therefore maintains higher absolute performance and degrades more gracefully. Both remain above the “no retrieval” baseline (~28–29%), but Self-CRAG stays substantially higher across the range.


In [85]:
len(answer["context"][0].content)

2

In [102]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


From the table you provided:

- CRAG: 27.2 TFLOPs per token; execution time 0.512 s per instance.  
- Self‑RAG: 26.5 → 132.4 TFLOPs per token (range); execution time 0.741 s per instance.

So CRAG runs faster (0.512 s vs 0.741 s) and has a stable cost of 27.2 TFLOPs/token. Self‑RAG can have a similar lower-bound cost (26.5 TFLOPs/token) but may require much more compute in the worst case (up to 132.4 TFLOPs/token). Note: these are rough estimates for the generation phase only; retrieval and data-processing are excluded.
